# 🔍 EDA — MercadoLibre: New vs Used Classification

**Challenge**: Predict if an item listed in MercadoLibre's Marketplace is **new** or **used**.  
**Target metric**: Accuracy ≥ 0.86 + secondary metric  
**Dataset**: 100,000 items from MLA (MercadoLibre Argentina)

---

In [1]:
import sys
sys.path.append('..')

import ast
import json
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import phik
from phik import phik_matrix
import warnings
warnings.filterwarnings('ignore')

from production.new_or_used import build_dataset

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.2f}'.format)


## 1. LIMPIEZA DE DATOS CRUDOS

### 1.1 Revisión de los datos

Analizamos estructura, tipo y contenido de las columnas

In [2]:
# Cargamos los datos con el split de la función otorgada en el challenge.
X_train, y_train, X_test, y_test = build_dataset()

print(f"Train set: {len(X_train):,} items")
print(f"Test set:  {len(X_test):,} items")
print(f"Total:     {len(X_train) + len(X_test):,} items")
print(f"\nTrain/Test ratio: {len(X_train)/(len(X_train)+len(X_test))*100:.1f}% / {len(X_test)/(len(X_train)+len(X_test))*100:.1f}%")

# Analizamos los tipos de datos y la estructura de los registros.
print('\n', "=" * 80, sep='')
print("ETRUCTURA DE LOS REGISTROS")
print("=" * 80, '\n')

sample = X_train[0]
for key in sorted(sample.keys()):
    val = sample[key]
    val_preview = repr(val)[:100]
    print(f"  {key:40s} ({type(val).__name__:10s})  →  {val_preview}")

# Creamos los dataframes y vemos los primeros registros para entender la estructura de los datos.
X_train = pd.DataFrame(X_train)
y_train = pd.Series(y_train)
X_test = pd.DataFrame(X_test)
y_test = pd.Series(y_test)

print('\n',"=" * 80, sep='')
print("HEAD DE TRAINING SET")
print("=" * 80)
X_train.head()

Train set: 90,000 items
Test set:  10,000 items
Total:     100,000 items

Train/Test ratio: 90.0% / 10.0%

ETRUCTURA DE LOS REGISTROS

  accepts_mercadopago                      (bool      )  →  True
  attributes                               (list      )  →  []
  automatic_relist                         (bool      )  →  False
  available_quantity                       (int       )  →  1
  base_price                               (float     )  →  80.0
  buying_mode                              (str       )  →  'buy_it_now'
  catalog_product_id                       (NoneType  )  →  None
  category_id                              (str       )  →  'MLA126406'
  condition                                (str       )  →  'new'
  coverage_areas                           (list      )  →  []
  currency_id                              (str       )  →  'ARS'
  date_created                             (str       )  →  '2015-09-05T20:42:53.000Z'
  deal_ids                                 (list    

,seller_address,warranty,sub_status,condition,deal_ids,base_price,shipping,non_mercado_pago_payment_methods,seller_id,variations,site_id,listing_type_id,price,attributes,buying_mode,tags,listing_source,parent_item_id,coverage_areas,category_id,descriptions,last_updated,international_delivery_mode,pictures,id,official_store_id,differential_pricing,accepts_mercadopago,original_price,currency_id,thumbnail,title,automatic_relist,date_created,secure_thumbnail,stop_time,status,video_id,catalog_product_id,subtitle,initial_quantity,start_time,permalink,sold_quantity,available_quantity
0,"{'country': {'name': 'Argentina', 'id': 'AR'},...",NaN,[],new,[],80.00,"{'local_pick_up': True, 'methods': [], 'tags':...","[{'description': 'Transferencia bancaria', 'id...",8208882349,[],MLA,bronze,80.00,[],buy_it_now,[dragged_bids_and_visits],,MLA6553902747,[],MLA126406,[{'id': 'MLA4695330653-912855983'}],2015-09-05T20:42:58.000Z,none,"[{'size': '500x375', 'secure_url': 'https://a2...",MLA4695330653,NaN,None,True,NaN,ARS,http://mla-s1-p.mlstatic.com/5386-MLA469533065...,Auriculares Samsung Originales Manos Libres Ca...,False,2015-09-05T20:42:53.000Z,https://a248.e.akamai.net/mla-s1-p.mlstatic.co...,1446669773000,active,NaN,NaN,None,1,1441485773000,http://articulo.mercadolibre.com.ar/MLA4695330...,0,1
1,"{'country': {'name': 'Argentina', 'id': 'AR'},...",NUESTRA REPUTACION,[],used,[],2650.00,"{'local_pick_up': True, 'methods': [], 'tags':...","[{'description': 'Transferencia bancaria', 'id...",8141699488,[],MLA,silver,2650.00,[],buy_it_now,[],,MLA7727150374,[],MLA10267,[{'id': 'MLA7160447179-930764806'}],2015-09-26T18:08:34.000Z,none,"[{'size': '499x334', 'secure_url': 'https://a2...",MLA7160447179,NaN,None,True,NaN,ARS,http://mla-s1-p.mlstatic.com/23223-MLA71604471...,Cuchillo Daga Acero Carbón Casco Yelmo Solinge...,False,2015-09-26T18:08:30.000Z,https://a248.e.akamai.net/mla-s1-p.mlstatic.co...,1448474910000,active,NaN,NaN,None,1,1443290910000,http://articulo.mercadolibre.com.ar/MLA7160447...,0,1
2,"{'country': {'name': 'Argentina', 'id': 'AR'},...",NaN,[],used,[],60.00,"{'local_pick_up': True, 'methods': [], 'tags':...","[{'description': 'Transferencia bancaria', 'id...",8386096505,[],MLA,bronze,60.00,[],buy_it_now,[dragged_bids_and_visits],,MLA6561247998,[],MLA1227,[{'id': 'MLA7367189936-916478256'}],2015-09-09T23:57:10.000Z,none,"[{'size': '375x500', 'secure_url': 'https://a2...",MLA7367189936,NaN,None,True,NaN,ARS,http://mla-s1-p.mlstatic.com/22076-MLA73671899...,"Antigua Revista Billiken, N° 1826, Año 1954",False,2015-09-09T23:57:07.000Z,https://a248.e.akamai.net/mla-s1-p.mlstatic.co...,1447027027000,active,NaN,NaN,None,1,1441843027000,http://articulo.mercadolibre.com.ar/MLA7367189...,0,1
3,"{'country': {'name': 'Argentina', 'id': 'AR'},...",NaN,[],new,[],580.00,"{'local_pick_up': True, 'methods': [], 'tags':...","[{'description': 'Transferencia bancaria', 'id...",5377752182,[],MLA,silver,580.00,[],buy_it_now,[],,NaN,[],MLA86345,[{'id': 'MLA9191625553-932309698'}],2015-10-05T16:03:50.306Z,none,"[{'size': '441x423', 'secure_url': 'https://a2...",MLA9191625553,NaN,None,True,NaN,ARS,http://mla-s2-p.mlstatic.com/183901-MLA9191625...,Alarma Guardtex Gx412 Seguridad Para El Automo...,False,2015-09-28T18:47:56.000Z,https://a248.e.akamai.net/mla-s2-p.mlstatic.co...,1449191596000,active,NaN,NaN,None,1,1443466076000,http://articulo.mercadolibre.com.ar/MLA9191625...,0,1
4,"{'country': {'name': 'Argentina', 'id': 'AR'},...",MI REPUTACION.,[],used,[],30.00,"{'local_pick_up': True, 'methods': [], 'tags':...","[{'description': 'Transferencia bancaria', 'id...",2938071313,[],MLA,bronze,30.00,[],buy_it_now,[dragged_bids_and_visits],,MLA3133256685,[],MLA41287,[{'id': 'MLA7787961817-902981678'}],2015-08-28T13:37:41.000Z,none,"[{'size': '375x500', 'secure_url': 'https://a2...",MLA7787961817,NaN,None,True,NaN,ARS,http://mla-s2-p.mlstatic.com/13595-MLA77879618...,Serenata - Jennifer Blake,False,2015-08-24T22:07:20.000Z,https://a248.e.akamai.net/mla-s2-p.mlstatic.

### 1.2 Identificacion y cálculo de datos faltantes.

Se identificaron muchos casos de datos faltantes ocultos las principales causas detectadas:
- Listas vacías `[]`
- Strings vacíos `""` o `"None"` (en cualquier capitalización)
- `None` de Python

Todos estos casos se convierten a `pd.NA` para poder calcular correctamente el porcentaje
de nulos por columna.

Una vez estandarizados, se identifican las columnas con **más del 95% de nulos** y se analiza
el balance de clases (`new` / `used`) en sus filas no nulas. El objetivo es determinar si la
*presencia* de un valor en esa columna es en sí misma una señal predictiva.

In [3]:
def clean_dataset_objects(df):

    cols_objeto = df.select_dtypes(include=['object']).columns

    for col in cols_objeto:
        df[col] = df[col].map(lambda x: pd.NA if (
            # Caso 1: Es una lista y está vacía
            (isinstance(x, list) and len(x) == 0) or 
            
            # Caso 2: Es un string y dice 'none' (independiente de mayúsculas/minúsculas)
            (isinstance(x, str) and x.strip().lower() in ['none', '']) or
            
            # Caso 3: Es el valor None de Python o ya es un NA (para consistencia)
            (x is None)
        ) else x)
            
    return df

X_train = clean_dataset_objects(X_train)
X_test = clean_dataset_objects(X_test)

# Columnas con más del 95% de nulos
high_null_cols = X_train.isnull().mean()
high_null_cols = high_null_cols[high_null_cols > 0.95].index.tolist()

# Balance de clases para cada una
for col in high_null_cols:
    mask = X_train[col].notna()
    print(f"\n── {col} ({mask.sum()} filas con valor) ──")
    print(y_train[mask].value_counts(normalize=True).round(3))


── sub_status (891 filas con valor) ──
new    0.55
used   0.45
Name: proportion, dtype: float64

── deal_ids (217 filas con valor) ──
new    0.98
used   0.02
Name: proportion, dtype: float64

── listing_source (0 filas con valor) ──
Series([], Name: proportion, dtype: float64)

── coverage_areas (0 filas con valor) ──
Series([], Name: proportion, dtype: float64)

── international_delivery_mode (0 filas con valor) ──
Series([], Name: proportion, dtype: float64)

── official_store_id (745 filas con valor) ──
new    0.97
used   0.03
Name: proportion, dtype: float64

── differential_pricing (0 filas con valor) ──
Series([], Name: proportion, dtype: float64)

── original_price (130 filas con valor) ──
new   1.00
Name: proportion, dtype: float64

── video_id (2676 filas con valor) ──
new    0.77
used   0.23
Name: proportion, dtype: float64

── catalog_product_id (7 filas con valor) ──
used   0.71
new    0.29
Name: proportion, dtype: float64

── subtitle (0 filas con valor) ──
Series([], Nam

### 1.3 Eliminación de columnas irrelevantes

Se eliminan las siguientes columnas por los motivos indicados:

### Totalidad de datos nulos / pocos datos no nutlos con target equilibrado
| Columna | Motivo |
|---|---|
|`sub_status` | Pocos registros y clases muy balanceadas | 
|`listing_source` | Totalidad de registros nulos |
|`coverage_areas` | Totalidad de registros nulos |
|`international_delivery_mode` | Totalidad de registros nulos |
|`differential_pricing` | Totalidad de registros nulos |
|`original_price` | Pocos registros |
|`catalog_product_id` | Pocos registros|
|`subtitle` | Totalidad de registros nulos |

### Alta cardinalidad / valores únicos
| Columna | Motivo |
|---|---|
| `id` | Identificador único por publicación |
| `permalink` | URL única por publicación |
| `thumbnail` | URL casi única por publicación |
| `secure_thumbnail` | URL casi única por publicación |
| `descriptions` | Texto casi único por publicación |
| `parent_item_id` | Identificador único |

### Baja varianza
| Columna | Motivo |
|---|---|
| `site_id` | 100% de los valores son `MLA` |
| `currency_id` | >99% de los valores son `ARS` |
| `status` | En listings activos (único escenario de inferencia) todos los valores son `active` |

### Variables temporales
| Columna | Motivo |
|---|---|
| `start_time` | Variable temporal sin poder predictivo directo |
| `stop_time` | Variable temporal sin poder predictivo directo |
| `last_updated` | Variable temporal sin poder predictivo directo |
| `date_created` | Variable temporal sin poder predictivo directo |

### Features descartadas por leakage temporal
`seller_id` y `title` fueron inicialmente candidatas para construir features de frecuencia 
(cantidad de publicaciones por vendedor, títulos repetidos). Sin embargo, su cálculo correcto 
requiere respetar la línea temporal — es decir, para cada publicación solo se podría considerar 
el historial **previo** a su fecha de creación. Dado que el split train/test es fijo y no está 
ordenado temporalmente, no es posible garantizar esta condición sin introducir leakage. 
Por lo tanto, se descartan.

In [4]:
cols_to_drop = ['sub_status', 'listing_source', 'coverage_areas', 
                'international_delivery_mode','differential_pricing',
                'original_price', 'catalog_product_id','subtitle', 'site_id', 
                'parent_item_id','descriptions', 'last_updated', 'id', 
                'currency_id','thumbnail', 'date_created', 'secure_thumbnail',
                'stop_time','status', 'start_time', 'permalink', 'seller_id',
                'title']

X_train.drop(columns=cols_to_drop, inplace=True)
X_test.drop(columns=cols_to_drop, inplace=True)

### 1.4 Elminamos la columna condition de train ya que es el target y no es una feature.

La función otorgada que separa los sets de entrenamiento y testeo no elmina de train el target.

In [5]:
X_train.drop(columns=['condition'], inplace=True)

## 2. Análisis del balance de clases en los sets de entrenamiento y test.

Se analizada que tan balanceadas están ambas clases en ambos datasets.

In [6]:
from collections import Counter

train_dist = Counter(y_train)
test_dist = Counter(y_test)

print(f"{'':20s} {'TRAIN':>10s} {'TEST':>10s}")
print("-" * 45)
for label in ['new', 'used']:
    t_count = train_dist[label]
    te_count = test_dist[label]
    t_pct = t_count / len(y_train) * 100
    te_pct = te_count / len(y_test) * 100
    print(f"  {label:<18s} {t_count:>6,} ({t_pct:.1f}%)  {te_count:>6,} ({te_pct:.1f}%)")
print("-" * 45)
print(f"  {'Total':<18s} {len(y_train):>6,}         {len(y_test):>6,}")
print(f"\n Se observa que ambos datasets están bastante balanceados, con una proporción similar de artículos nuevos y usados en ambos sets.")

                          TRAIN       TEST
---------------------------------------------
  new                48,352 (53.7%)   5,406 (54.1%)
  used               41,648 (46.3%)   4,594 (45.9%)
---------------------------------------------
  Total              90,000         10,000

 Se observa que ambos datasets están bastante balanceados, con una proporción similar de artículos nuevos y usados en ambos sets.


## 3. Feature Engineering

Ahora que entendemos la data cruda, construimos un DataFrame con features aplanadas y derivadas.  
Lo hacemos **por separado** para train y test para mantener la integridad de los splits.

### 3.1 Análisis de la columna 'seller_address'

La columna contiene un diccionario con la siguiente estructura:
```json
{
    "country": {"name": "pais", "id": "id_pais"},
    "state":   {"name": "provincia", "id": "id_provincia"},
    "city":    {"name": "ciudad", "id": ""}
}
```

In [7]:
# Verifico si todos coinciden exactamente tanto en train como en test.

def check_structure(df):
    keys = {'country', 'state', 'city'}
    is_consistent = df['seller_address'].apply(lambda x: set(x.keys()) == keys).all()
    if is_consistent:
        print("¡Todos tienen la misma estructura!")
    else:
        print("Hay filas con estructuras diferentes.")


check_structure(X_train)
check_structure(X_test)

¡Todos tienen la misma estructura!
¡Todos tienen la misma estructura!


In [8]:
# Chequeo los valores únicos de cada clave mostrando los 5 más frecuentes.

def chequeo_valor_unicos(df):
    cols_interes = ['country', 'state', 'city']

    for col in cols_interes:
        top_valores = df['seller_address'].str[col].str['name'].value_counts().head(5)
        print(f"\nDistribución de {col}:")
        print(top_valores)
print('-----Train-----')       
chequeo_valor_unicos(X_train)
print('\n-----Test-----')       
chequeo_valor_unicos(X_test)

-----Train-----

Distribución de country:
seller_address
Argentina    89999
                 1
Name: count, dtype: int64

Distribución de state:
seller_address
Capital Federal    52143
Buenos Aires       31482
Santa Fe            2398
Córdoba             1727
Mendoza              400
Name: count, dtype: int64

Distribución de city:
seller_address
CABA               3708
Buenos Aires       3104
Capital Federal    3053
Palermo            2995
Caballito          2675
Name: count, dtype: int64

-----Test-----

Distribución de country:
seller_address
Argentina    9998
                2
Name: count, dtype: int64

Distribución de state:
seller_address
Capital Federal    5711
Buenos Aires       3531
Santa Fe            274
Córdoba             181
Mendoza              50
Name: count, dtype: int64

Distribución de city:
seller_address
CABA               390
Palermo            360
Capital Federal    349
Buenos Aires       316
Caballito          292
Name: count, dtype: int64



Se extraen `state` y `city` usando el campo `name` de cada nivel.

**Criterios de selección:**
- Se descarta `country` porque casi el 100% de los valores corresponden a Argentina, sin variación útil para el modelo.
- Se descarta el campo `id` de cada nivel porque presenta muchos valores vacíos y es redundante con `name`.

In [9]:
def process_address_data(df, column_name='seller_address'):
    """
    Extrae country, state y city de una columna JSON y las integra al DataFrame.
    """
    # 1. Aseguramos que el índice esté limpio para evitar desalineación en el concat
    df = df.reset_index(drop=True)
    
    # 2. Aplanamos el JSON
    # errors='ignore' ayuda si hay filas con valores inesperados
    df_temp = pd.json_normalize(df[column_name])
    
    # 3. Seleccionamos y renombramos las columnas deseadas
    # Usamos .get() o filtramos con cuidado por si alguna key no existe en el set
    cols_map = {
        'state.name': 'state',
        'city.name': 'city'
    }
    
    # Filtramos solo las columnas que efectivamente existen tras el normalize
    existing_cols = [c for c in cols_map.keys() if c in df_temp.columns]
    df_names = df_temp[existing_cols].rename(columns=cols_map)
    
    # 4. Concatenamos y eliminamos la original
    df_final = pd.concat([df, df_names], axis=1)
    df_final = df_final.drop(columns=[column_name])
    
    return df_final

X_train = process_address_data(X_train)
X_test = process_address_data(X_test)

### 3.2 Análisis de la columna 'shipping'

La columna contiene un diccionario con información de envío. Se extraen tres features y se descarta el resto:

| Feature | Descripción | 
|---|---|
| `shipping_local_pick_up` | Si el vendedor ofrece retiro en persona | 
| `shipping_free` | Si el envío es gratuito | 
| `shipping_mode` | Modalidad de envío (`me2`, `me1`, `custom`, `not_specified`) |

Se descartan `methods`, `tags` y `dimensions` por tener valores casi siempre vacíos o nulos.

In [10]:
def parse_shipping(df, col='shipping'):
    parsed = df[col].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

    df = df.copy()
    df['shipping_local_pick_up'] = parsed.apply(lambda x: x.get('local_pick_up', False)).astype(int)
    df['shipping_free']          = parsed.apply(lambda x: x.get('free_shipping', False)).astype(int)
    df['shipping_mode']          = parsed.apply(lambda x: x.get('mode', 'not_specified'))

    return df.drop(columns=[col])

X_train = parse_shipping(X_train)
X_test  = parse_shipping(X_test)

### 3.3 Columnas con listas de objetos

Las columnas `variations`, `tags`, `attributes` y `pictures` contienen listas de objetos JSON.
En lugar de intentar aplanar su contenido — que es heterogéneo y de alta cardinalidad — se extrae
únicamente la cantidad de elementos como feature numérica.

| Feature | Descripción |
|---|---|
| `variations_count` | Cantidad de variantes del producto (talle, color, etc.) | 
| `tags_count` | Cantidad de etiquetas asociadas a la publicación | 
| `attributes_count` | Cantidad de atributos técnicos del producto | 
| `pictures_count` | Cantidad de fotos de la publicación | 

Para `pictures` se extrae además `pictures_max_area`, el área en píxeles de la foto de mayor
resolución disponible, como proxy de la calidad de las imágenes subidas.

In [11]:
def count_list_col(df, col):
    def count(val):
        if val is None or (isinstance(val, float) and pd.isna(val)):
            return 0
        if isinstance(val, list):
            return len(val)
        if isinstance(val, str):
            if val.strip() == '':
                return 0
            try:
                parsed = ast.literal_eval(val)
                return len(parsed) if isinstance(parsed, list) else 0
            except:
                return 0
        return 0

    df = df.copy()
    df[f'{col}_count'] = df[col].apply(count)
    return df.drop(columns=[col])

for col in ['non_mercado_pago_payment_methods', 'variations',
            'tags', 'attributes']:
    X_train = count_list_col(X_train, col)
    X_test  = count_list_col(X_test, col)

In [12]:
def parse_pictures(df, col='pictures'):
    def process(val):
        if val is None or (isinstance(val, float) and pd.isna(val)):
            return 0, 0
        if isinstance(val, str):
            if val.strip() == '':
                return 0, 0
            try:
                val = ast.literal_eval(val)
            except:
                return 0, 0
        if isinstance(val, list):
            count = len(val)
            max_area = 0
            for pic in val:
                try:
                    w, h = pic.get('max_size', '0x0').split('x')
                    max_area = max(max_area, int(w) * int(h))
                except:
                    continue
            return count, max_area
        return 0, 0

    results = df[col].apply(process)
    df = df.copy()
    df['pictures_count']    = results.apply(lambda x: x[0])
    df['pictures_max_area'] = results.apply(lambda x: x[1])
    return df.drop(columns=[col])

X_train = parse_pictures(X_train)
X_test  = parse_pictures(X_test)

### 3.4 Análisis y procesamiento de la variable `warranty`

La columna contiene texto libre describiendo la garantía del producto. Al ser texto no estructurado,
se normalizó y clasificó en categorías mediante un pipeline de NLP:

1. **Normalización**: limpieza de texto (lowercase, eliminación de caracteres especiales, etc.)
2. **Etiquetado**: se etiquetaron de forma automática utilizando un LLM 1000 muestras representativas como conjunto de entrenamiento
3. **Clasificación**: se entrenó un clasificador usando embeddings multilingües 
   (`paraphrase-multilingual-MiniLM-L12-v2`) + `SGDClassifier` para asignar cada valor a una categoría

Las categorías finales son:

| Categoría | Descripción |
|---|---|
| `NaN` | Sin información de garantía |
| `Sin garantía` | Explícitamente sin garantía |
| `Hasta 3 meses` | Garantía de corto plazo |
| `4–6 meses` | Garantía de mediano plazo |
| `7–12 meses` | Garantía cercana al año |
| `1 año` | Garantía de un año exacto |
| `2 años` | Garantía de dos años |
| `3–5 años` | Garantía extendida |
| `5+ años` | Garantía de muy largo plazo |
| `Garantía de fábrica` | Garantía del fabricante sin duración explícita |
| `Con garantía` | Mención genérica de garantía |
| `Reputación del vendedor` | Garantía basada en la reputación del vendedor |
| `Garantía de satisfacción` | Garantía de devolución o satisfacción |

In [13]:
from production.warranty_classifier import classify_warranties
df_labeled =pd.read_csv('../warranty_samples.csv')

# Varios dfs — el modelo se entrena una sola vez
warranty_train, warranty_test = classify_warranties(df_labeled, X_train[['warranty']], X_test[['warranty']])

X_train['warranty'] = warranty_train['warranty_label']
X_test['warranty'] = warranty_test['warranty_label']

[train] 1042 ejemplos | 10 clases


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/17 [00:00<?, ?it/s]

[train] F1 macro (5-fold): 0.785 ± 0.026
[predict] 90000 filas | confianza media: 0.912
[predict] 10000 filas | confianza media: 0.907


### 3.4 Columnas con pocos registros y más de una categoría.

Convierto en booleanas las colunmas  deal_ids, official_store_id y video_id por poseer pocos registros y más de una categoría lo que incorpora ruido en el entrenamiento.


In [14]:
def convert_to_bool(df, cols):
    for col in cols:
        df[col] = np.where(df[col].isnull(), 0, 1)
    return df

cols = ['deal_ids', 'official_store_id', 'video_id']

X_train = convert_to_bool(X_train, cols)
X_test  = convert_to_bool(X_test, cols)

## 4. Realizo cálculos de estadísticos sobre las variables numéricas

In [15]:
# Build DataFrames
df_train = X_train.copy()
df_train['condition'] = y_train
df_train['target'] = (df_train['condition'] == 'new').astype(int)

df_test = X_test.copy() 
df_test['condition'] = y_test
df_test['target'] = (df_test['condition'] == 'new').astype(int)

print(f"Train DataFrame: {df_train.shape}")
print(f"Test DataFrame:  {df_test.shape}")

Train DataFrame: (90000, 27)
Test DataFrame:  (10000, 27)


In [16]:
# Describe by condition WITHIN train
print("=" * 80)
print("TRAIN SET — Statistics by Condition")
print("=" * 80)

desc_new = df_train[df_train['condition'] == 'new'].describe().T[['mean', '50%', 'std']].round(2)
desc_used = df_train[df_train['condition'] == 'used'].describe().T[['mean', '50%', 'std']].round(2)

comp_train = pd.concat([desc_new, desc_used], axis=1, keys=['NEW', 'USED'])
comp_train[('DIFF', 'mean_diff_%')] = (
    (desc_new['mean'] - desc_used['mean']) / desc_used['mean'].replace(0, np.nan) * 100
).round(1)
comp_train

TRAIN SET — Statistics by Condition


NEW                        \
                                            mean       50%         std   
deal_ids                                    0.00      0.00        0.07   
base_price                              49672.21    350.00 10106294.84   
price                                   49672.37    350.00 10106294.84   
official_store_id                           0.01      0.00        0.12   
video_id                                    0.04      0.00        0.20   
initial_quantity                           63.27      2.00      570.43   
sold_quantity                               4.25      0.00       45.85   
available_quantity                         62.81      2.00      570.08   
shipping_local_pick_up                      0.81      1.00        0.39   
shipping_free                               0.05      0.00        0.22   
non_mercado_pago_payment_methods_count      1.73      2.00        1.38   
variations_count                            0.28      0.00        1.45   
tags_count                                  0.77      1.00        0.47   
attributes_count                            0.58      0.00        4.08   
pictures_count                              3.02      3.00        2.08   
pictures_max_area                      673573.54 758208.00   433812.36   
target                                      1.00      1.00        0.00   

                                            USED                        \
                                            mean        50%        std   
deal_ids                                    0.00       0.00       0.01   
base_price                              67265.08     150.00 7743572.43   
price                                   67265.13     150.00 7743572.43   
official_store_id                           0.00       0.00       0.02   
video_id                                    0.01       0.00       0.12   
initial_quantity                            2.09       1.00      58.31   
sold_quantity                               0.10       0.00       4.98   
available_quantity                          2.07       1.00      58.20   
shipping_local_pick_up                      0.78       1.00       0.42   
shipping_free                               0.00       0.00       0.07   
non_mercado_pago_payment_methods_count      1.41       1.00       1.42   
variations_count                            0.08       0.00       0.29   
tags_count                                  0.76       1.00       0.44   
attributes_count                            1.87       0.00      10.07   
pictures_count                              2.82       2.00       2.13   
pictures_max_area                      809846.12 1042800.00  380916.72   
target                                      0.00       0.00       0.00   

                                              DIFF  
                                       mean_diff_%  
deal_ids                                       NaN  
base_price                                  -26.20  
price                                       -26.20  
official_store_id                              NaN  
video_id                                    300.00  
initial_quantity                           2927.30  
sold_quantity                              4150.00  
available_quantity                         2934.30  
shipping_local_pick_up                        3.80  
shipping_free                                  NaN  
non_mercado_pago_payment_methods_count       22.70  
variations_count                            250.00  
tags_count                                    1.30  
attributes_count                            -69.00  
pictures_count                                7.10  
pictures_max_area                           -16.80  
target                                         NaN

In [17]:
def describe_by_condition(df_train, df_test, condition_col='condition'):
    
    for condition in ['new', 'used']:
        print("=" * 80)
        print(f"CONDITION: {condition.upper()}")
        print("=" * 80)
        
        tr = df_train[df_train[condition_col] == condition]
        te = df_test[df_test[condition_col] == condition]
        
        # --- Tabla 1: resumen ---
        summary_stats = ['mean', 'std', '50%', 'min', 'max']
        desc_tr = tr.describe(percentiles=[.5]).T[summary_stats].round(2)
        desc_te = te.describe(percentiles=[.5]).T[summary_stats].round(2)
        
        summary = pd.concat([desc_tr, desc_te], axis=1, keys=['TRAIN', 'TEST'])
        summary[('DIFF', 'mean_%')] = (
            (desc_tr['mean'] - desc_te['mean']) / desc_te['mean'].replace(0, np.nan) * 100
        ).round(1)
        
        print("\n── Summary ──")
        display(summary)
        
        # --- Tabla 2: quantiles ---
        quantile_stats = ['25%', '50%', '75%', '90%', '95%', '99%']
        desc_tr_q = tr.describe(percentiles=[.25, .5, .75, .90, .95, .99]).T[quantile_stats].round(2)
        desc_te_q = te.describe(percentiles=[.25, .5, .75, .90, .95, .99]).T[quantile_stats].round(2)
        
        quantiles = pd.concat([desc_tr_q, desc_te_q], axis=1, keys=['TRAIN', 'TEST'])
        
        print("\n── Quantiles ──")
        display(quantiles)

describe_by_condition(df_train, df_test)

CONDITION: NEW

── Summary ──


TRAIN                             \
                                            mean         std       50%  min   
deal_ids                                    0.00        0.07      0.00 0.00   
base_price                              49672.21 10106294.84    350.00 0.84   
price                                   49672.37 10106294.84    350.00 0.84   
official_store_id                           0.01        0.12      0.00 0.00   
video_id                                    0.04        0.20      0.00 0.00   
initial_quantity                           63.27      570.43      2.00 1.00   
sold_quantity                               4.25       45.85      0.00 0.00   
available_quantity                         62.81      570.08      2.00 1.00   
shipping_local_pick_up                      0.81        0.39      1.00 0.00   
shipping_free                               0.05        0.22      0.00 0.00   
non_mercado_pago_payment_methods_count      1.73        1.38      2.00 0.00   
variations_count                            0.28        1.45      0.00 0.00   
tags_count                                  0.77        0.47      1.00 0.00   
attributes_count                            0.58        4.08      0.00 0.00   
pictures_count                              3.02        2.08      3.00 0.00   
pictures_max_area                      673573.54   433812.36 758208.00 0.00   
target                                      1.00        0.00      1.00 1.00   

                                                          TEST            \
                                                 max      mean       std   
deal_ids                                        1.00      0.00      0.07   
base_price                             2222222222.00   2622.55  19523.09   
price                                  2222222222.00   2622.55  19523.09   
official_store_id                               1.00      0.01      0.11   
video_id                                        1.00      0.04      0.21   
initial_quantity                             9999.00     66.12    570.85   
sold_quantity                                6065.00      5.52    120.94   
available_quantity                           9999.00     65.75    570.66   
shipping_local_pick_up                          1.00      0.81      0.39   
shipping_free                                   1.00      0.05      0.23   
non_mercado_pago_payment_methods_count         12.00      1.74      1.44   
variations_count                               50.00      0.27      1.41   
tags_count                                      2.00      0.78      0.47   
attributes_count                               81.00      0.55      3.88   
pictures_count                                 36.00      3.02      2.03   
pictures_max_area                         1440000.00 672291.40 431259.89   
target                                          1.00      1.00      0.00   

                                                                    DIFF  
                                             50%  min        max  mean_%  
deal_ids                                    0.00 0.00       1.00     NaN  
base_price                                358.60 1.00  720000.00 1794.00  
price                                     358.60 1.00  720000.00 1794.00  
official_store_id                           0.00 0.00       1.00    0.00  
video_id                                    0.00 0.00       1.00    0.00  
initial_quantity                            2.00 1.00    9999.00   -4.30  
sold_quantity                               0.00 0.00    8676.00  -23.00  
available_quantity                          2.00 1.00    9999.00   -4.50  
shipping_local_pick_up                      1.00 0.00       1.00    0.00  
shipping_free                               0.00 0.00       1.00    0.00  
non_mercado_pago_payment_methods_count      2.00 0.00      12.00   -0.60  
variations_count                            0.00 0.00      36.00    3.70  
tags_count                                  1.


── Quantiles ──


TRAIN                       \
                                             25%       50%        75%   
deal_ids                                    0.00      0.00       0.00   
base_price                                140.00    350.00     995.11   
price                                     140.00    350.00     995.59   
official_store_id                           0.00      0.00       0.00   
video_id                                    0.00      0.00       0.00   
initial_quantity                            1.00      2.00       9.00   
sold_quantity                               0.00      0.00       1.00   
available_quantity                          1.00      2.00       9.00   
shipping_local_pick_up                      1.00      1.00       1.00   
shipping_free                               0.00      0.00       0.00   
non_mercado_pago_payment_methods_count      0.00      2.00       3.00   
variations_count                            0.00      0.00       0.00   
tags_count                                  0.00      1.00       1.00   
attributes_count                            0.00      0.00       0.00   
pictures_count                              1.00      3.00       4.00   
pictures_max_area                      237596.50 758208.00 1080000.00   
target                                      1.00      1.00       1.00   

                                                                         \
                                              90%        95%        99%   
deal_ids                                     0.00       0.00       0.00   
base_price                                3096.90    6493.60   33900.00   
price                                     3098.84    6498.45   33900.00   
official_store_id                            0.00       0.00       1.00   
video_id                                     0.00       0.00       1.00   
initial_quantity                            31.90     100.00     999.00   
sold_quantity                                5.00      13.00      77.00   
available_quantity                          30.00     100.00     999.00   
shipping_local_pick_up                       1.00       1.00       1.00   
shipping_free                                0.00       1.00       1.00   
non_mercado_pago_payment_methods_count       3.00       3.00       4.00   
variations_count                             0.00       1.00       7.00   
tags_count                                   1.00       1.00       2.00   
attributes_count                             2.00       2.00       3.00   
pictures_count                               6.00       6.00       7.00   
pictures_max_area                      1125600.00 1340400.00 1440000.00   
target                                       1.00       1.00       1.00   

                                            TEST                       \
                                             25%       50%        75%   
deal_ids                                    0.00      0.00       0.00   
base_price                                140.00    358.60    1022.60   
price                                     140.00    358.60    1022.60   
official_store_id                           0.00      0.00       0.00   
video_id                                    0.00      0.00       0.00   
initial_quantity                            1.00      2.00      10.00   
sold_quantity                               0.00      0.00       1.00   
available_quantity                          1.00      2.00       9.00   
shipping_local_pick_up                      1.00      1.00       1.00   
shipping_free                               0.00      0.00       0.00   
non_mercado_pago_payment_methods_count      0.00      2.00       3.00   
variations_count                            0.00      0.00       0.00   
tags_count                                  1.00      1.00       1.00   
attributes_count                            0.00      0.00       0.00   
pictures_count                              1.00      3.00   

CONDITION: USED

── Summary ──


TRAIN                             \
                                            mean        std        50%  min   
deal_ids                                    0.00       0.01       0.00 0.00   
base_price                              67265.08 7743572.43     150.00 1.00   
price                                   67265.13 7743572.43     150.00 1.00   
official_store_id                           0.00       0.02       0.00 0.00   
video_id                                    0.01       0.12       0.00 0.00   
initial_quantity                            2.09      58.31       1.00 1.00   
sold_quantity                               0.10       4.98       0.00 0.00   
available_quantity                          2.07      58.20       1.00 1.00   
shipping_local_pick_up                      0.78       0.42       1.00 0.00   
shipping_free                               0.00       0.07       0.00 0.00   
non_mercado_pago_payment_methods_count      1.41       1.42       1.00 0.00   
variations_count                            0.08       0.29       0.00 0.00   
tags_count                                  0.76       0.44       1.00 0.00   
attributes_count                            1.87      10.07       0.00 0.00   
pictures_count                              2.82       2.13       2.00 0.00   
pictures_max_area                      809846.12  380916.72 1042800.00 0.00   
target                                      0.00       0.00       0.00 0.00   

                                                          TEST            \
                                                 max      mean       std   
deal_ids                                        1.00      0.00      0.00   
base_price                             1111111111.00   7627.04  52389.79   
price                                  1111111111.00   7627.05  52389.79   
official_store_id                               1.00      0.00      0.01   
video_id                                        1.00      0.02      0.12   
initial_quantity                             9999.00      1.25      3.32   
sold_quantity                                 982.00      0.07      0.54   
available_quantity                           9999.00      1.24      3.30   
shipping_local_pick_up                          1.00      0.78      0.41   
shipping_free                                   1.00      0.01      0.07   
non_mercado_pago_payment_methods_count         12.00      1.43      1.41   
variations_count                               10.00      0.08      0.30   
tags_count                                      2.00      0.75      0.45   
attributes_count                               81.00      2.15     10.81   
pictures_count                                 18.00      2.84      2.15   
pictures_max_area                         1440000.00 801265.88 390559.16   
target                                          0.00      0.00      0.00   

                                                                     DIFF  
                                              50%  min        max  mean_%  
deal_ids                                     0.00 0.00       0.00     NaN  
base_price                                 150.00 1.00 1499000.00  781.90  
price                                      150.00 1.00 1499000.00  781.90  
official_store_id                            0.00 0.00       1.00     NaN  
video_id                                     0.00 0.00       1.00  -50.00  
initial_quantity                             1.00 1.00      98.00   67.20  
sold_quantity                                0.00 0.00      18.00   42.90  
available_quantity                           1.00 1.00      98.00   66.90  
shipping_local_pick_up                       1.00 0.00       1.00    0.00  
shipping_free                                0.00 0.00       1.00 -100.00  
non_mercado_pago_payment_methods_count       1.00 0.00      12.00   -1.40  
variations_count                             0.00 0.00       7.00    0.00  
tags_count                      


── Quantiles ──


TRAIN                        \
                                             25%        50%        75%   
deal_ids                                    0.00       0.00       0.00   
base_price                                 65.00     150.00     546.22   
price                                      65.00     150.00     548.25   
official_store_id                           0.00       0.00       0.00   
video_id                                    0.00       0.00       0.00   
initial_quantity                            1.00       1.00       1.00   
sold_quantity                               0.00       0.00       0.00   
available_quantity                          1.00       1.00       1.00   
shipping_local_pick_up                      1.00       1.00       1.00   
shipping_free                               0.00       0.00       0.00   
non_mercado_pago_payment_methods_count      0.00       1.00       2.00   
variations_count                            0.00       0.00       0.00   
tags_count                                  1.00       1.00       1.00   
attributes_count                            0.00       0.00       0.00   
pictures_count                              1.00       2.00       4.00   
pictures_max_area                      476100.00 1042800.00 1080000.00   
target                                      0.00       0.00       0.00   

                                                                         \
                                              90%        95%        99%   
deal_ids                                     0.00       0.00       0.00   
base_price                                3000.00    9903.25  193154.00   
price                                     3000.00    9903.25  193154.00   
official_store_id                            0.00       0.00       0.00   
video_id                                     0.00       0.00       1.00   
initial_quantity                             1.00       1.00       5.00   
sold_quantity                                0.00       0.00       1.00   
available_quantity                           1.00       1.00       5.00   
shipping_local_pick_up                       1.00       1.00       1.00   
shipping_free                                0.00       0.00       0.00   
non_mercado_pago_payment_methods_count       3.00       3.00       5.00   
variations_count                             0.00       1.00       1.00   
tags_count                                   1.00       1.00       1.00   
attributes_count                             2.00       2.00      65.00   
pictures_count                               6.00       6.00      11.00   
pictures_max_area                      1080000.00 1164000.00 1430400.00   
target                                       0.00       0.00       0.00   

                                            TEST                        \
                                             25%        50%        75%   
deal_ids                                    0.00       0.00       0.00   
base_price                                 65.00     150.00     550.00   
price                                      65.00     150.00     550.00   
official_store_id                           0.00       0.00       0.00   
video_id                                    0.00       0.00       0.00   
initial_quantity                            1.00       1.00       1.00   
sold_quantity                               0.00       0.00       0.00   
available_quantity                          1.00       1.00       1.00   
shipping_local_pick_up                      1.00       1.00       1.00   
shipping_free                               0.00       0.00       0.00   
non_mercado_pago_payment_methods_count      0.00       1.00       2.00   
variations_count                            0.00       0.00       0.00   
tags_count                                  0.00       1.00       1.00   
attributes_count                            0.00       0.00       0.00   
pictures_count            

### 4.1 Identificación de outliers

#### `initial_quantity`, `sold_quantity`, `available_quantity`

Las tres variables presentan distribuciones extremadamente sesgadas:

- El 75% de los valores se concentra en 9 o menos
- Los máximos alcanzan 9999 (`initial_quantity`, `available_quantity`) y 6065 (`sold_quantity`) 
- Los desvíos estándar superan ampliamente a las medias

Sin embargo, se intuye que estos valores extremos **no son errores** — reflejan vendedores profesionales con
stock masivo, que en su mayoría corresponden a productos nuevos. Eliminarlos implicaría perder
información predictiva valiosa. Se podrá manejar esta varianza al escalar los datos.


---

#### `price` y `base_price`

Presentan una situación más extrema:

- Mediana de 150-350 pero máximo de 2,222,222,222
- Desvío estándar de ~9 millones
- La media de `new` en train (49k) vs test (2.6k) sugiere que los outliers están concentrados
  en train, lo que introduce inconsistencia entre conjuntos

A diferencia de las variables de cantidad, el valor máximo de 2,222,222,222 **es claramente un
error de carga** y no un precio real. Al analizar los valores mayores se identifican claramente outliers.
La decisión es topear a un máximo 5,000,000 de pesos.

In [18]:
X_train['price'].sort_values(ascending=False).head(30)

53241   2222222222.00
10891   1111111111.00
36137   1111111111.00
62854    123456789.00
69028    112111111.00
15783     11111111.00
27166     11111111.00
4201      11111111.00
56269      9000000.00
43769      8888888.00
78341      6500000.00
1273       5330000.00
79066      3956000.00
71341      2648840.00
3190       2500000.00
36548      2300000.00
60010      2050000.00
41416      2004105.00
82609      1919000.00
73029      1800000.00
72807      1720000.00
27234      1700000.00
19192      1690000.00
4222       1628005.00
25158      1559237.00
81463      1525000.00
81857      1500000.00
81501      1500000.00
83902      1499000.00
38626      1484000.00
Name: price, dtype: float64

In [19]:
CAP_PRICE = 5_000_000

for col in ['price', 'base_price']:
    X_train[col] = np.log1p(X_train[col].clip(upper=CAP_PRICE))
    X_test[col]  = np.log1p(X_test[col].clip(upper=CAP_PRICE))

In [20]:
X_train['non_mercado_pago_payment_methods_count'].value_counts()

non_mercado_pago_payment_methods_count
0     27531
2     22688
3     22030
1     14594
4      2282
11      187
5       175
6       126
7       100
9        91
8        74
12       64
10       58
Name: count, dtype: int64

## 5.Análisis de correlación de las

Utilizo la matriz de correlación de phik ya que funciona para **todas las combinaciones** de tipos:
- Numérica ↔ Numérica
- Numérica ↔ Categórica  
- Categórica ↔ Categórica

Calculamos sobre el **train set** para evitar data leakage.

In [21]:
phik_cols = [
    'target',
    # Categorical
    'warranty', 'listing_type_id', 'buying_mode', 'shipping_mode', 'category_id',
    'state',
    # Numeric
    'base_price', 'price', 'initial_quantity', 'sold_quantity', 'available_quantity',
    'pictures_count', 'pictures_max_area', 'non_mercado_pago_payment_methods_count',
    'tags_count', 'attributes_count', 'variations_count',
    # Binary
    'shipping_free', 'shipping_local_pick_up', 'accepts_mercadopago',
    'automatic_relist', 'video_id', 'deal_ids', 'official_store_id',
]

phik_corr = df_train[phik_cols].phik_matrix()

fig = px.imshow(
    phik_corr, text_auto='.2f',
    color_continuous_scale='RdBu_r', zmin=0, zmax=1,
    aspect='auto',
    title='φk Correlation Matrix — TRAIN SET'
)
fig.update_layout(template='plotly_white', height=900, width=1000, font=dict(size=10))
fig.show()

interval columns not set, guessing: ['target', 'base_price', 'price', 'initial_quantity', 'sold_quantity', 'available_quantity', 'pictures_count', 'pictures_max_area', 'non_mercado_pago_payment_methods_count', 'tags_count', 'attributes_count', 'variations_count', 'shipping_free', 'shipping_local_pick_up', 'video_id', 'deal_ids', 'official_store_id']


In [22]:
# φk with target — ranked
target_phik = phik_corr['target'].drop('target').sort_values(ascending=False)

fig = go.Figure(data=[
    go.Bar(
        x=target_phik.values, y=target_phik.index, orientation='h',
        marker_color=px.colors.sample_colorscale('Viridis', target_phik.values / target_phik.values.max()),
        text=[f'{v:.3f}' for v in target_phik.values], textposition='outside'
    )
])
fig.update_layout(
    title='φk Correlation with Target — TRAIN SET',
    template='plotly_white', height=600, width=800,
    xaxis_title='φk Correlation', yaxis=dict(autorange='reversed'),
    font=dict(size=12)
)
fig.show()

### 4.1 Análisis de los resultados

### Variables mejor correlacionadas con el target

| Variable | φk | Interpretación |
|---|---|---|
| `category_id` | 0.80 | Señal más fuerte — la categoría del producto determina en gran medida si es nuevo o usado |
| `listing_type_id` | 0.48 | El tipo de publicación refleja el nivel de inversión del vendedor |
| `pictures_max_area` | 0.38 | Mayor resolución de fotos asociada a productos nuevos |
| `warranty` | 0.30 | Productos nuevos tienden a tener garantías más formales |
| `automatic_relist` | 0.29 | Vendedores de nuevo usan más el relisting automático |
| `non_mercado_pago_payment_methods_count` | 0.25 | Señal moderada |
| `shipping_free` | 0.21 | Vendedores profesionales ofrecen más envío gratis |

### Multicolinealidad detectada

Se identificaron tres pares de variables con correlación φk = 1.00, lo que indica redundancia
perfecta. En cada caso se elimina una de las dos:

| Par | Decisión |
|---|---|
| `initial_quantity` / `available_quantity` | Se elimina `available_quantity` |
| `base_price` / `price` | Se elimina `base_price` |
| `buying_mode` / `accepts_mercadopago` | Se elimina `accepts_mercadopago` |

### Variables de baja señal

Las variables con baja señal serían suceptibles de ser eliminadas, pero decido delegarle esta tarea al modelo y evidenciarlo en el feature importance.

In [23]:
cols_to_drop = [
    'available_quantity',  # correlación 1.00 con initial_quantity
    'base_price',          # correlación 1.00 con price
    'accepts_mercadopago', # correlación 1.00 con buying_mode
]

X_train.drop(columns=cols_to_drop, inplace=True)
X_test.drop(columns=cols_to_drop, inplace=True)

## 7. Deep Dive: Top Features (Train vs Test)

Basándonos en las correlaciones, profundizamos en las features más discriminantes.
Comparamos train y test en cada gráfico.

In [24]:
# Top N categorías por condición
N = 20

# Conteo por categoría y condición
cat_counts = df_train.groupby(['category_id', 'condition']).size().unstack(fill_value=0)
cat_counts['total'] = cat_counts.sum(axis=1)
cat_counts['pct_new'] = (cat_counts['new'] / cat_counts['total'] * 100).round(1)
cat_counts['pct_used'] = (cat_counts['used'] / cat_counts['total'] * 100).round(1)

# Top N por volumen total
top_cats = cat_counts.nlargest(N, 'total')

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=['Volumen por categoría (Top 20)',
                                    '% New vs Used por categoría (Top 20)'])

# --- Subplot 1: volumen absoluto ---
fig.add_trace(go.Bar(
    y=top_cats.index, x=top_cats['new'],
    name='new', orientation='h',
    marker_color='#d62728'
), row=1, col=1)

fig.add_trace(go.Bar(
    y=top_cats.index, x=top_cats['used'],
    name='used', orientation='h',
    marker_color='#1f77b4'
), row=1, col=1)

# --- Subplot 2: proporción ---
fig.add_trace(go.Bar(
    y=top_cats.index, x=top_cats['pct_new'],
    name='% new', orientation='h',
    marker_color='#d62728', showlegend=False
), row=1, col=2)

fig.add_trace(go.Bar(
    y=top_cats.index, x=top_cats['pct_used'],
    name='% used', orientation='h',
    marker_color='#1f77b4', showlegend=False
), row=1, col=2)

fig.update_layout(
    barmode='stack',
    template='plotly_white',
    height=600, width=1200,
    title='Distribución de category_id por condición — Top 20 categorías',
    legend=dict(orientation='h', y=-0.1)
)
fig.update_xaxes(title_text='Cantidad', row=1, col=1)
fig.update_xaxes(title_text='Porcentaje', row=1, col=2)

fig.show()

In [25]:
binary_feats = ['deal_ids', 'official_store_id', 'video_id', 'automatic_relist', 'shipping_local_pick_up', 'shipping_free', 'accepts_mercadopago']

results = []
for split_name, df_split in [('Train', df_train), ('Test', df_test)]:
    for feat in binary_feats:
        subset_1 = df_split[df_split[feat] == 1]
        pct_new = (subset_1['condition'] == 'new').mean() * 100 if len(subset_1) > 0 else 0
        results.append({'split': split_name, 'feature': feat, 'pct_new_when_1': pct_new, 
                        'count_1': len(subset_1)})

res_df = pd.DataFrame(results)

fig = go.Figure()
for split, color, offset in [('Train', '#2979FF', -0.2), ('Test', '#FF6D00', 0.2)]:
    subset = res_df[res_df['split'] == split].sort_values('pct_new_when_1', ascending=True)
    fig.add_trace(go.Bar(
        y=subset['feature'], x=subset['pct_new_when_1'], orientation='h',
        name=split, marker_color=color,
        text=[f"{v:.1f}%" for v in subset['pct_new_when_1']], textposition='outside'
    ))

fig.add_shape(type='line', x0=53.8, x1=53.8, y0=-0.5, y1=len(binary_feats)-0.5,
              line=dict(color='red', width=2, dash='dash'))
fig.add_annotation(x=55, y=len(binary_feats)-0.5, text='Baseline', showarrow=False,
                   yshift=15, font=dict(color='red', size=11))

fig.update_layout(
    title='% New when Feature=1: Train vs Test',
    template='plotly_white', height=450, width=900, barmode='group',
    xaxis_title='% New', xaxis_range=[0, 110], font=dict(size=12)
)
fig.show()

print("→ Si las barras train y test son similares, las features son estables entre splits")

→ Si las barras train y test son similares, las features son estables entre splits


In [28]:
X_train.to_parquet('../data/processed/X_train.parquet', index=False)
X_test.to_parquet('../data/processed/X_test.parquet', index=False)
y_train.to_frame().to_parquet('../data/processed/y_train.parquet', index=False)
y_test.to_frame().to_parquet('../data/processed/y_test.parquet', index=False)

In [2]:
! pip install optuna


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 5.8 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 263.9/263.9 kB 6.7 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 8.4 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.5/78.5 kB 4.7 MB/s eta 0:00:00

[notice] A new release of pip is available: 23.1.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
